# Audio Transcription Pipeline with AssemblyAI

This notebook demonstrates how to build an audio transcription pipeline using:
- **AssemblyAI** for speech-to-text transcription
- **Haystack** for document processing and pipeline orchestration
- **SentenceTransformers** for document embedding

## Features:
- Transcribes audio files from URLs or local paths
- Splits transcriptions into manageable chunks
- Generates embeddings for semantic search
- Stores processed documents in memory for quick access

## Requirements:
- AssemblyAI API key (set in `.env` file as `ASSEMBLYAI_API_KEY`)
- Internet connection for downloading audio and models

In [3]:
import os
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from assemblyai_haystack.transcriber import AssemblyAITranscriber
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get API key from environment variables
ASSEMBLYAI_API_KEY = os.getenv("ASSEMBLYAI_API_KEY")

if not ASSEMBLYAI_API_KEY:
    raise ValueError("ASSEMBLYAI_API_KEY not found in environment variables. Please add it to your .env file.")

print(f"API Key loaded: {'*' * (len(ASSEMBLYAI_API_KEY) - 4) + ASSEMBLYAI_API_KEY[-4:]}")

# Initialize components
document_store = InMemoryDocumentStore()
transcriber = AssemblyAITranscriber(api_key=ASSEMBLYAI_API_KEY)
document_splitter = DocumentSplitter(
    split_by="word",
    split_length=150,
    split_overlap=50
)
document_writer = DocumentWriter(document_store)
document_embedder = SentenceTransformersDocumentEmbedder()

# Build pipeline
preprocessing_pipeline = Pipeline()
preprocessing_pipeline.add_component(instance=transcriber, name="transcriber")
preprocessing_pipeline.add_component(instance=document_splitter, name="document_splitter")
preprocessing_pipeline.add_component(instance=document_embedder, name="document_embedder")
preprocessing_pipeline.add_component(instance=document_writer, name="document_writer")

# Connect pipeline components
preprocessing_pipeline.connect("transcriber.transcription", "document_splitter")
preprocessing_pipeline.connect("document_splitter", "document_embedder")
preprocessing_pipeline.connect("document_embedder", "document_writer")

print("Pipeline built successfully!")

API Key loaded: ****************************0561

Pipeline built successfully!Pipeline built successfully!



In [4]:
# Run the audio transcription pipeline
file_path = "https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3"

print(f"Starting transcription of: {file_path}")
print("This may take a few minutes...")

try:
    result = preprocessing_pipeline.run({
        "transcriber": {"file_path": file_path}
    })
    
    print("✅ Transcription completed successfully!")
    print(f"Documents stored: {document_store.count_documents()}")
    
    # Show a sample of the transcribed content
    if document_store.count_documents() > 0:
        sample_doc = list(document_store.filter_documents({}))[0]
        print(f"\nSample transcription content:")
        print(f"Content preview: {sample_doc.content[:200]}...")
        print(f"Metadata: {sample_doc.meta}")
        
except Exception as e:
    print(f"❌ Error during transcription: {str(e)}")
    print("Please check your API key and internet connection.")

Starting transcription of: https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3
This may take a few minutes...

This may take a few minutes...


c:\Users\777kr\Desktop\Cerebrus AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\777kr\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling

✅ Transcription completed successfully!
Documents stored: 9

Sample transcription content:
Documents stored: 9

Sample transcription content:
Content preview: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, the air quality warn...
Metadata: {'transcript_id': 'fe278fe2-9d8d-4f39-874d-9628cca69252', 'audio_url': 'https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3', 'source_id': '3403d693188b9ae9fb74fef6332766b82cc4ee1dd4025ff2be423c80a46e7893', 'page_number': 1, 'split_id': 0, 'split_idx_start': 0, '_split_overlap': [{'doc_id': '231f821b56df5cf9b2e2b91b2b324c70a430c0bbe66a51a3271800775e0315e1', 'range': (0, 300)}]}

Content preview: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US Skylines from Maine to Maryland to Minnesota are gray and smoggy. And in some places, th

In [5]:
# Explore the transcribed documents
print("=== Document Analysis ===")
print(f"Total documents in store: {document_store.count_documents()}")

if document_store.count_documents() > 0:
    documents = list(document_store.filter_documents({}))
    
    print(f"\nDocument breakdown:")
    for i, doc in enumerate(documents[:5]):  # Show first 5 documents
        print(f"\nDocument {i+1}:")
        print(f"  Content length: {len(doc.content)} characters")
        print(f"  Content: {doc.content[:150]}...")
        if doc.meta:
            print(f"  Metadata keys: {list(doc.meta.keys())}")
    
    if len(documents) > 5:
        print(f"\n... and {len(documents) - 5} more documents")
        
    # Show total content length
    total_content = sum(len(doc.content) for doc in documents)
    print(f"\nTotal transcribed content: {total_content} characters")
    
else:
    print("No documents found. Please run the transcription first.")

=== Document Analysis ===
Total documents in store: 9

Document breakdown:

Document 1:
  Content length: 910 characters
  Content: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US Skylines from Maine to Maryland to Minnesota are gray an...
  Metadata keys: ['transcript_id', 'audio_url', 'source_id', 'page_number', 'split_id', 'split_idx_start', '_split_overlap']

Document 2:
Total documents in store: 9

Document breakdown:

Document 1:
  Content length: 910 characters
  Content: Smoke from hundreds of wildfires in Canada is triggering air quality alerts throughout the US Skylines from Maine to Maryland to Minnesota are gray an...
  Metadata keys: ['transcript_id', 'audio_url', 'source_id', 'page_number', 'split_id', 'split_idx_start', '_split_overlap']

Document 2:
  Content length: 886 characters
  Content: there's a couple of things. The season has been pretty dry already, and then the fact that we're getting hit in the US is because ther

# Enhanced Audio Processor - MEW.py Features in Haystack

Now let's create a comprehensive audio processor that replicates your `mew.py` functionality using Haystack components:

## Missing Features from Current Implementation:
1. **Speaker Diarization** (identifying different speakers)
2. **Timestamp Information** (start/end times for segments)
3. **Smart Chunking with Speaker Awareness**
4. **Citation System** for audio chunks
5. **Advanced Metadata** (confidence, duration, speaker count)
6. **Custom Document Structure** for audio content
7. **Batch Processing** capabilities

In [ ]:
import sys
sys.path.append('..')  # Add parent directory to path

import hashlib
from datetime import datetime
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
from pathlib import Path
import assemblyai as aai
from haystack import component, Document, Pipeline
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore

@dataclass
class AudioDocumentChunk:
    """Enhanced document chunk for audio with speaker and timing information"""
    content: str
    source_file: str
    source_type: str = 'audio'
    chunk_index: int = 0
    start_char: Optional[int] = None
    end_char: Optional[int] = None
    start_timestamp: Optional[float] = None
    end_timestamp: Optional[float] = None
    speakers: List[str] = None
    confidence: Optional[float] = None
    chunk_id: str = ""
    metadata: Dict[str, Any] = None
    
    def __post_init__(self):
        if not self.chunk_id:
            self.chunk_id = self._generate_chunk_id()
        if self.metadata is None:
            self.metadata = {}
        if self.speakers is None:
            self.speakers = []
    
    def _generate_chunk_id(self) -> str:
        content_hash = hashlib.md5(self.content.encode()).hexdigest()[:8]
        return f"audio_{self.chunk_index}_{content_hash}"
    
    def get_citation_info(self) -> Dict[str, Any]:
        citation = {
            'source': self.source_file,
            'type': self.source_type,
            'chunk_id': self.chunk_id,
            'chunk_index': self.chunk_index,
            'speakers': self.speakers,
            'timestamp_range': f"{self._format_time(self.start_timestamp)}-{self._format_time(self.end_timestamp)}" if self.start_timestamp and self.end_timestamp else None
        }
        citation.update(self.metadata)
        return citation
    
    def _format_time(self, seconds: Optional[float]) -> str:
        if seconds is None:
            return "00:00"
        minutes = int(seconds // 60)
        seconds = int(seconds % 60)
        return f"{minutes:02d}:{seconds:02d}"

@component
class SmartAudioProcessor:
    """
    Enhanced Audio Processor that replicates mew.py functionality:
    - Speaker diarization
    - Smart chunking with speaker awareness
    - Timestamp preservation
    - Citation system
    - Advanced metadata
    """
    
    def __init__(self, api_key: str, chunk_size: int = 1000, chunk_overlap: int = 100):
        self.api_key = api_key
        aai.settings.api_key = api_key
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        
        self.supported_formats = {
            '.mp3', '.wav', '.m4a', '.aac', '.ogg', 
            '.flac', '.wma', '.opus', '.mp4', '.mov', '.avi'
        }
    
    @component.output_types(documents=List[Document])
    def run(self, sources: List[str], 
            enable_speaker_diarization: bool = True,
            enable_auto_punctuation: bool = True,
            language_code: str = "en") -> Dict[str, List[Document]]:
        
        all_documents = []
        
        for source in sources:
            try:
                # Configure AssemblyAI with advanced features
                config = aai.TranscriptionConfig(
                    speaker_labels=enable_speaker_diarization,
                    punctuate=enable_auto_punctuation,
                    language_code=language_code,
                    format_text=True,
                    boost_param="low"  # Enhanced accuracy
                )
                
                transcriber = aai.Transcriber(config=config)
                transcript = transcriber.transcribe(source)
                
                if transcript.status == aai.TranscriptStatus.error:
                    print(f"❌ Transcription failed for {source}: {transcript.error}")
                    continue
                
                print(f"✅ Transcription completed for: {Path(source).name if source.startswith('http') else source}")
                
                # Process transcript into smart chunks
                documents = self._create_smart_audio_chunks(transcript, source)
                all_documents.extend(documents)
                
            except Exception as e:
                print(f"❌ Error processing {source}: {str(e)}")
                continue
        
        return {"documents": all_documents}
    
    def _create_smart_audio_chunks(self, transcript: aai.Transcript, source: str) -> List[Document]:
        source_name = Path(source).name if not source.startswith('http') else source.split('/')[-1]
        
        # Base metadata from transcript
        base_metadata = {
            'duration_seconds': transcript.audio_duration,
            'confidence': transcript.confidence,
            'transcription_id': transcript.id,
            'word_count': len(transcript.text.split()) if transcript.text else 0,
            'character_count': len(transcript.text) if transcript.text else 0,
            'processed_at': datetime.now().isoformat()
        }
        
        documents = []
        
        # Check if we have speaker diarization
        if hasattr(transcript, 'utterances') and transcript.utterances:
            documents = self._create_speaker_aware_chunks(
                transcript.utterances, source_name, base_metadata
            )
        else:
            # Fallback to simple chunking
            documents = self._create_simple_chunks(
                transcript.text, source_name, base_metadata
            )
        
        print(f"Created {len(documents)} speaker-aware chunks from {source_name}")
        return documents
    
    def _create_speaker_aware_chunks(self, utterances, source_file: str, base_metadata: Dict) -> List[Document]:
        documents = []
        current_text = ""
        current_speakers = []
        current_timestamps = []
        chunk_index = 0
        start_char = 0
        
        for utterance in utterances:
            speaker_label = f"Speaker {utterance.speaker}"
            
            # Convert milliseconds to seconds
            start_time = utterance.start / 1000.0
            end_time = utterance.end / 1000.0
            
            # Format with timestamp
            timestamp_str = f"[{self._format_milliseconds(utterance.start)}]"
            speaker_text = f"{timestamp_str} {speaker_label}: {utterance.text}\n"
            
            # Check if we should create a new chunk
            if len(current_text + speaker_text) > self.chunk_size and current_text:
                # Create chunk with speaker metadata
                chunk_metadata = base_metadata.copy()
                chunk_metadata.update({
                    'speakers': list(set(current_speakers)),
                    'speaker_count': len(set(current_speakers)),
                    'start_timestamp': min(current_timestamps) if current_timestamps else None,
                    'end_timestamp': max(current_timestamps) if current_timestamps else None,
                })
                
                # Generate chunk ID
                content_hash = hashlib.md5(current_text.encode()).hexdigest()[:8]
                chunk_id = f"audio_{chunk_index}_{content_hash}"
                
                chunk_metadata.update({
                    'chunk_id': chunk_id,
                    'chunk_index': chunk_index,
                    'start_char': start_char,
                    'end_char': start_char + len(current_text) - 1,
                    'citation': {
                        'source': source_file,
                        'type': 'audio',
                        'chunk_id': chunk_id,
                        'speakers': list(set(current_speakers)),
                        'timestamp_range': f"{self._format_time(min(current_timestamps))}-{self._format_time(max(current_timestamps))}" if current_timestamps else None
                    }
                })
                
                document = Document(
                    content=current_text.strip(),
                    meta=chunk_metadata
                )
                documents.append(document)
                
                # Handle overlap
                overlap_text = current_text[-self.chunk_overlap:] if self.chunk_overlap > 0 else ""
                current_text = overlap_text + speaker_text
                start_char += len(current_text) - len(overlap_text) - len(speaker_text)
                chunk_index += 1
                
                current_speakers = [speaker_label]
                current_timestamps = [start_time, end_time]
            else:
                current_text += speaker_text
                current_speakers.append(speaker_label)
                current_timestamps.extend([start_time, end_time])
        
        # Handle remaining text
        if current_text.strip():
            chunk_metadata = base_metadata.copy()
            chunk_metadata.update({
                'speakers': list(set(current_speakers)),
                'speaker_count': len(set(current_speakers)),
                'start_timestamp': min(current_timestamps) if current_timestamps else None,
                'end_timestamp': max(current_timestamps) if current_timestamps else None,
            })
            
            content_hash = hashlib.md5(current_text.encode()).hexdigest()[:8]
            chunk_id = f"audio_{chunk_index}_{content_hash}"
            
            chunk_metadata.update({
                'chunk_id': chunk_id,
                'chunk_index': chunk_index,
                'start_char': start_char,
                'end_char': start_char + len(current_text) - 1,
                'citation': {
                    'source': source_file,
                    'type': 'audio',
                    'chunk_id': chunk_id,
                    'speakers': list(set(current_speakers)),
                    'timestamp_range': f"{self._format_time(min(current_timestamps))}-{self._format_time(max(current_timestamps))}" if current_timestamps else None
                }
            })
            
            document = Document(
                content=current_text.strip(),
                meta=chunk_metadata
            )
            documents.append(document)
        
        return documents
    
    def _create_simple_chunks(self, text: str, source_file: str, base_metadata: Dict) -> List[Document]:
        if not text.strip():
            return []
        
        documents = []
        start = 0
        chunk_index = 0
        
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            
            # Smart boundary detection
            if end < len(text):
                last_period = text.rfind('.', start, end)
                last_newline = text.rfind('\n', start, end)
                boundary = max(last_period, last_newline)
                if boundary > start + self.chunk_size * 0.5:
                    end = boundary + 1
            
            chunk_text = text[start:end].strip()
            
            if chunk_text:
                chunk_metadata = base_metadata.copy()
                chunk_metadata.update({
                    'speakers': ['Unknown Speaker'],
                    'speaker_count': 1
                })
                
                content_hash = hashlib.md5(chunk_text.encode()).hexdigest()[:8]
                chunk_id = f"audio_{chunk_index}_{content_hash}"
                
                chunk_metadata.update({
                    'chunk_id': chunk_id,
                    'chunk_index': chunk_index,
                    'start_char': start,
                    'end_char': end - 1,
                    'citation': {
                        'source': source_file,
                        'type': 'audio',
                        'chunk_id': chunk_id,
                        'speakers': ['Unknown Speaker']
                    }
                })
                
                document = Document(
                    content=chunk_text,
                    meta=chunk_metadata
                )
                documents.append(document)
                chunk_index += 1
            
            start = max(start + self.chunk_size - self.chunk_overlap, end)
        
        return documents
    
    def _format_milliseconds(self, ms: int) -> str:
        seconds = ms // 1000
        minutes = seconds // 60
        seconds = seconds % 60
        return f"{minutes:02d}:{seconds:02d}"
    
    def _format_time(self, seconds: Optional[float]) -> str:
        if seconds is None:
            return "00:00"
        minutes = int(seconds // 60)
        secs = int(seconds % 60)
        return f"{minutes:02d}:{secs:02d}"

print("✅ SmartAudioProcessor component created!")

In [ ]:
# Test the Smart Audio Processor
print("Testing Smart Audio Processor...")

# Create enhanced processor with speaker diarization
smart_audio_processor = SmartAudioProcessor(
    api_key=ASSEMBLYAI_API_KEY,
    chunk_size=800,  # Smaller chunks for better speaker granularity
    chunk_overlap=150
)

# Test with the same audio file
test_audio_url = "https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3"

print(f"Processing audio with Smart Audio Processor...")
print("This includes speaker diarization and advanced chunking...")

try:
    smart_result = smart_audio_processor.run(
        sources=[test_audio_url],
        enable_speaker_diarization=True,
        enable_auto_punctuation=True,
        language_code="en"
    )
    
    smart_documents = smart_result["documents"]
    print(f"\n✅ Smart processing completed!")
    print(f"Created {len(smart_documents)} enhanced chunks")
    
    # Analyze the first few chunks
    print(f"\n=== SMART CHUNK ANALYSIS ===")
    for i, doc in enumerate(smart_documents[:3]):
        print(f"\nSmart Chunk {i+1}:")
        print(f"  Content preview: {doc.content[:150]}...")
        print(f"  Speakers: {doc.meta.get('speakers', [])}")
        print(f"  Speaker count: {doc.meta.get('speaker_count', 0)}")
        print(f"  Timestamp range: {doc.meta.get('citation', {}).get('timestamp_range', 'N/A')}")
        print(f"  Chunk ID: {doc.meta.get('chunk_id', 'N/A')}")
        print(f"  Confidence: {doc.meta.get('confidence', 'N/A')}")
    
except Exception as e:
    print(f"❌ Error in smart processing: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
# Create Enhanced Audio Pipeline with Smart Processing
print("=== ENHANCED AUDIO PIPELINE ===")

# Create new document store for enhanced processing
enhanced_audio_store = InMemoryDocumentStore()

# Create enhanced pipeline with smart processor
enhanced_audio_pipeline = Pipeline()
enhanced_audio_pipeline.add_component("smart_audio_processor", smart_audio_processor)
enhanced_audio_pipeline.add_component("document_writer", DocumentWriter(document_store=enhanced_audio_store))
enhanced_audio_pipeline.connect("smart_audio_processor", "document_writer")

print("Running Enhanced Audio Pipeline with Smart Processing...")

try:
    enhanced_result = enhanced_audio_pipeline.run({
        "smart_audio_processor": {
            "sources": [test_audio_url],
            "enable_speaker_diarization": True,
            "enable_auto_punctuation": True,
            "language_code": "en"
        }
    })
    
    print(f"\n✅ Enhanced Pipeline Results:")
    print(f"Documents written: {enhanced_result['document_writer']['documents_written']}")
    print(f"Total documents in enhanced store: {enhanced_audio_store.count_documents()}")
    
    # Compare with original approach
    print(f"\n=== COMPARISON ===")
    print(f"Original Basic Pipeline: {document_store.count_documents()} documents")
    print(f"Enhanced Smart Pipeline: {enhanced_audio_store.count_documents()} documents")
    
    # Show citation capabilities
    if enhanced_audio_store.count_documents() > 0:
        sample_doc = list(enhanced_audio_store.filter_documents({}))[0]
        citation = sample_doc.meta.get('citation', {})
        
        print(f"\n=== CITATION DEMO ===")
        print("Enhanced Citation Information:")
        for key, value in citation.items():
            print(f"  {key}: {value}")
        
        print(f"\nSample Citation String:")
        print(f"Source: {citation.get('source', 'N/A')}")
        print(f"Speakers: {', '.join(citation.get('speakers', []))}")
        print(f"Timestamp: {citation.get('timestamp_range', 'N/A')}")
        print(f"Type: Audio Transcript")
    
except Exception as e:
    print(f"❌ Error in enhanced pipeline: {str(e)}")
    import traceback
    traceback.print_exc()

In [ ]:
# Advanced Features Demo: Batch Processing and Summary
print("=== ADVANCED FEATURES DEMO ===")

@component  
class AudioSummaryGenerator:
    """Component that generates summaries of audio transcriptions"""
    
    def __init__(self, api_key: str):
        self.api_key = api_key
        aai.settings.api_key = api_key
    
    @component.output_types(summary=Dict[str, Any])
    def run(self, audio_url: str) -> Dict[str, Dict[str, Any]]:
        try:
            config = aai.TranscriptionConfig(
                speaker_labels=True,
                summarization=True,
                summary_model="informative",
                summary_type="bullets"
            )
            
            transcriber = aai.Transcriber(config=config)
            transcript = transcriber.transcribe(audio_url)
            
            if transcript.status == aai.TranscriptStatus.error:
                return {"summary": {"error": transcript.error}}
            
            summary_info = {
                'id': transcript.id,
                'duration_seconds': transcript.audio_duration,
                'confidence': transcript.confidence,
                'word_count': len(transcript.text.split()) if transcript.text else 0,
                'character_count': len(transcript.text) if transcript.text else 0,
                'summary': getattr(transcript, 'summary', 'Not available'),
                'speaker_count': len(set(u.speaker for u in transcript.utterances)) if hasattr(transcript, 'utterances') and transcript.utterances else 1,
                'language_detected': getattr(transcript, 'language_code', 'en')
            }
            
            return {"summary": summary_info}
            
        except Exception as e:
            return {"summary": {"error": str(e)}}

# Test summary generation
summary_generator = AudioSummaryGenerator(api_key=ASSEMBLYAI_API_KEY)

print("Generating audio summary...")
summary_result = summary_generator.run(audio_url=test_audio_url)
summary = summary_result["summary"]

if "error" not in summary:
    print(f"\n✅ Audio Summary Generated:")
    print(f"  Duration: {summary['duration_seconds']} seconds")
    print(f"  Confidence: {summary['confidence']:.2f}")
    print(f"  Word Count: {summary['word_count']}")
    print(f"  Speaker Count: {summary['speaker_count']}")
    print(f"  Language: {summary['language_detected']}")
    
    if summary.get('summary') != 'Not available':
        print(f"\n  Summary:")
        print(f"  {summary['summary']}")
else:
    print(f"❌ Summary generation failed: {summary['error']}")

print(f"\n=== BATCH PROCESSING DEMO ===")
# Demonstrate batch processing capability
batch_urls = [
    "https://github.com/AssemblyAI-Examples/audio-examples/raw/main/20230607_me_canadian_wildfires.mp3"
    # Add more URLs here for real batch processing
]

print(f"Processing {len(batch_urls)} audio files in batch...")
batch_result = smart_audio_processor.run(
    sources=batch_urls,
    enable_speaker_diarization=True,
    enable_auto_punctuation=True
)

batch_documents = batch_result["documents"]
print(f"✅ Batch processing completed: {len(batch_documents)} total chunks from {len(batch_urls)} files")

In [ ]:
import os
import sys
from typing import List, Dict, Any, Optional, Union
from pathlib import Path
from dataclasses import dataclass, field
import logging
from urllib.parse import urlparse
import io

from haystack import component, Document, default_from_dict, default_to_dict
from haystack.core.serialization import default_from_dict, default_to_dict

try:
    import assemblyai as aai
    ASSEMBLYAI_AVAILABLE = True
except ImportError:
    ASSEMBLYAI_AVAILABLE = False
    aai = None

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class AudioProcessingConfig:
    """Configuration for AssemblyAI audio processing features."""
    
    # Core transcription settings
    language_code: Optional[str] = "en"
    model: str = "best"  # 'best', 'nano', 'conformer-2'
    
    # Speaker features
    speaker_labels: bool = True
    speakers_expected: Optional[int] = None
    
    # Content analysis
    sentiment_analysis: bool = True
    entity_detection: bool = True
    iab_categories: bool = True  # Topic detection
    content_safety: bool = True
    content_safety_confidence: int = 80
    auto_highlights: bool = True
    
    # Audio enhancement
    noise_reduction: bool = True
    automatic_punctuation: bool = True
    format_text: bool = True
    filter_profanity: bool = False
    
    # Privacy and redaction
    redact_pii: bool = False
    redact_pii_policies: List[str] = field(default_factory=lambda: [
        "credit_card_number", "email_address", "person_name", "phone_number"
    ])
    redact_pii_audio: bool = False
    
    # Advanced features
    custom_spelling: Dict[str, List[str]] = field(default_factory=dict)
    custom_vocabulary: List[str] = field(default_factory=list)
    boost_param: str = "low"  # 'low', 'default', 'high'
    
    # Output formats
    include_utterances: bool = True
    include_sentences: bool = True
    include_paragraphs: bool = True
    auto_chapters: bool = True
    summarization: bool = True
    summary_model: str = "informative"  # 'informative', 'conversational', 'catchy'
    summary_type: str = "bullets"  # 'bullets', 'gist', 'headline', 'paragraph'

@component
class AssemblyAITranscriber:
    """
    A comprehensive Haystack component for AssemblyAI speech-to-text transcription.
    
    This component provides full access to AssemblyAI's advanced features including
    speaker diarization, content analysis, sentiment analysis, and more.
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        config: Optional[AudioProcessingConfig] = None,
        polling_interval: float = 3.0
    ):
        """
        Initialize the AssemblyAI Transcriber component.
        
        :param api_key: AssemblyAI API key. If None, uses ASSEMBLYAI_API_KEY env var
        :param config: Audio processing configuration
        :param polling_interval: Polling interval for checking transcription status
        """
        
        if not ASSEMBLYAI_AVAILABLE:
            raise ImportError(
                "assemblyai package is required. Install with: pip install assemblyai"
            )
        
        # Get API key
        api_key = api_key or os.getenv("ASSEMBLYAI_API_KEY")
        if not api_key:
            raise ValueError(
                "AssemblyAI API key required. Set ASSEMBLYAI_API_KEY env var or pass api_key parameter."
            )
        
        # Set API key and polling interval
        aai.settings.api_key = api_key
        aai.settings.polling_interval = polling_interval
        
        # Initialize configuration
        self.config = config or AudioProcessingConfig()
        self.api_key = api_key
        self.polling_interval = polling_interval
        
        # Initialize transcriber
        self.transcriber = aai.Transcriber()
        
        logger.info("AssemblyAI Transcriber initialized successfully")

    def _create_transcription_config(self) -> 'aai.TranscriptionConfig':
        """Create AssemblyAI TranscriptionConfig from our config."""
        
        config = aai.TranscriptionConfig()
        
        # Basic settings
        if self.config.language_code:
            config.language_code = self.config.language_code
        
        # Speaker settings
        config.speaker_labels = self.config.speaker_labels
        if self.config.speakers_expected:
            config.speakers_expected = self.config.speakers_expected
        
        # Content analysis
        config.sentiment_analysis = self.config.sentiment_analysis
        config.entity_detection = self.config.entity_detection
        config.iab_categories = self.config.iab_categories
        config.content_safety = self.config.content_safety
        config.content_safety_confidence = self.config.content_safety_confidence
        config.auto_highlights = self.config.auto_highlights
        
        # Audio enhancement
        config.format_text = self.config.format_text
        config.punctuate = self.config.automatic_punctuation
        config.filter_profanity = self.config.filter_profanity
        
        # Privacy settings
        if self.config.redact_pii:
            config.redact_pii = True
            config.redact_pii_policies = [
                getattr(aai.PIIRedactionPolicy, policy, policy)
                for policy in self.config.redact_pii_policies
                if hasattr(aai.PIIRedactionPolicy, policy)
            ]
            config.redact_pii_audio = self.config.redact_pii_audio
        
        # Custom vocabulary
        if self.config.custom_spelling:
            config.set_custom_spelling(self.config.custom_spelling)
        
        if self.config.custom_vocabulary:
            config.word_boost = self.config.custom_vocabulary
            if hasattr(aai, 'BoostParam'):
                config.boost_param = getattr(aai.BoostParam, self.config.boost_param, self.config.boost_param)
        
        # Chapter and summarization settings
        config.auto_chapters = self.config.auto_chapters
        if self.config.summarization:
            config.summarization = True
            if hasattr(aai, 'SummarizationModel'):
                config.summary_model = getattr(aai.SummarizationModel, self.config.summary_model, self.config.summary_model)
            if hasattr(aai, 'SummarizationType'):
                config.summary_type = getattr(aai.SummarizationType, self.config.summary_type, self.config.summary_type)
        
        return config

    @component.output_types(documents=List[Document])
    def run(
        self, 
        sources: List[Union[str, Path, bytes]]
    ) -> Dict[str, List[Document]]:
        """
        Transcribe audio files or URLs using AssemblyAI.
        
        :param sources: List of audio file paths, URLs, or bytes
        :return: Dictionary with 'documents' key containing transcribed documents
        """
        
        documents = []
        
        for source in sources:
            try:
                # Handle different source types
                if isinstance(source, bytes):
                    # Upload bytes to AssemblyAI
                    upload_url = self.transcriber.upload_file(source)
                    source_url = upload_url
                    source_name = "uploaded_audio"
                elif isinstance(source, (str, Path)):
                    source_str = str(source)
                    if self._is_url(source_str):
                        source_url = source_str
                        source_name = Path(urlparse(source_str).path).name or "web_audio"
                    else:
                        # Local file - upload to AssemblyAI
                        with open(source, 'rb') as f:
                            upload_url = self.transcriber.upload_file(f.read())
                        source_url = upload_url
                        source_name = Path(source).name
                else:
                    raise ValueError(f"Unsupported source type: {type(source)}")
                
                # Create transcription config
                transcript_config = self._create_transcription_config()
                
                # Transcribe
                logger.info(f"Starting transcription for: {source_name}")
                transcript = self.transcriber.transcribe(source_url, transcript_config)
                
                if transcript.status == aai.TranscriptStatus.error:
                    logger.error(f"Transcription failed for {source_name}: {transcript.error}")
                    continue
                
                # Extract comprehensive content
                content_parts = [f"# Transcription: {source_name}\n"]
                
                # Main transcript
                if transcript.text:
                    content_parts.append(f"## Full Transcript\n{transcript.text}\n")
                
                # Speaker-labeled transcript
                if self.config.speaker_labels and hasattr(transcript, 'utterances') and transcript.utterances:
                    content_parts.append("## Speaker Transcript\n")
                    for utterance in transcript.utterances:
                        content_parts.append(f"**Speaker {utterance.speaker}** ({utterance.start}ms - {utterance.end}ms): {utterance.text}\n")
                
                # Auto chapters
                if self.config.auto_chapters and hasattr(transcript, 'chapters') and transcript.chapters:
                    content_parts.append("## Chapters\n")
                    for i, chapter in enumerate(transcript.chapters):
                        content_parts.append(f"### Chapter {i+1}: {chapter.headline}\n")
                        content_parts.append(f"**Time**: {chapter.start}ms - {chapter.end}ms\n")
                        content_parts.append(f"**Summary**: {chapter.summary}\n")
                        content_parts.append(f"**Gist**: {chapter.gist}\n\n")
                
                # Summary
                if self.config.summarization and hasattr(transcript, 'summary') and transcript.summary:
                    content_parts.append(f"## Summary\n{transcript.summary}\n")
                
                # Create comprehensive metadata
                metadata = {
                    "source": source_name,
                    "transcript_id": transcript.id,
                    "audio_duration_seconds": getattr(transcript, 'audio_duration_seconds', None),
                    "language_code": self.config.language_code,
                    "confidence": getattr(transcript, 'confidence', None),
                    "audio_url": source_url if isinstance(source, str) and self._is_url(str(source)) else None
                }
                
                # Add analysis results to metadata
                if self.config.sentiment_analysis and hasattr(transcript, 'sentiment_analysis'):
                    metadata['sentiment_analysis'] = self._extract_sentiment_data(transcript.sentiment_analysis)
                
                if self.config.entity_detection and hasattr(transcript, 'entities'):
                    metadata['entities'] = self._extract_entity_data(transcript.entities)
                
                if self.config.iab_categories and hasattr(transcript, 'iab_categories'):
                    metadata['topics'] = self._extract_topic_data(transcript.iab_categories)
                
                if self.config.content_safety and hasattr(transcript, 'content_safety'):
                    metadata['content_safety'] = self._extract_content_safety_data(transcript.content_safety)
                
                if self.config.auto_highlights and hasattr(transcript, 'auto_highlights'):
                    metadata['highlights'] = self._extract_highlights_data(transcript.auto_highlights)
                
                # Create main document
                main_document = Document(
                    content="\n".join(content_parts),
                    meta=metadata
                )
                documents.append(main_document)
                
                # Create additional structured documents if requested
                self._add_structured_documents(transcript, documents, metadata)
                
                logger.info(f"Successfully transcribed {source_name} - Generated {len(documents)} documents")
                
            except Exception as e:
                logger.error(f"Error processing source {source}: {str(e)}")
                continue
        
        return {"documents": documents}
    
    def _add_structured_documents(self, transcript, documents: List[Document], base_metadata: Dict):
        """Add structured documents (sentences, paragraphs) if requested."""
        
        if self.config.include_sentences and hasattr(transcript, 'get_sentences'):
            try:
                sentences = transcript.get_sentences()
                for i, sentence in enumerate(sentences):
                    sentence_doc = Document(
                        content=sentence.text,
                        meta={
                            **base_metadata,
                            "content_type": "sentence",
                            "sentence_index": i,
                            "start_time": sentence.start,
                            "end_time": sentence.end
                        }
                    )
                    documents.append(sentence_doc)
            except Exception as e:
                logger.warning(f"Failed to extract sentences: {e}")
        
        if self.config.include_paragraphs and hasattr(transcript, 'get_paragraphs'):
            try:
                paragraphs = transcript.get_paragraphs()
                for i, paragraph in enumerate(paragraphs):
                    paragraph_doc = Document(
                        content=paragraph.text,
                        meta={
                            **base_metadata,
                            "content_type": "paragraph", 
                            "paragraph_index": i,
                            "start_time": paragraph.start,
                            "end_time": paragraph.end
                        }
                    )
                    documents.append(paragraph_doc)
            except Exception as e:
                logger.warning(f"Failed to extract paragraphs: {e}")
    
    def _is_url(self, string: str) -> bool:
        """Check if a string is a valid URL."""
        try:
            result = urlparse(string)
            return all([result.scheme, result.netloc])
        except Exception:
            return False
    
    def _extract_sentiment_data(self, sentiment_results) -> List[Dict]:
        """Extract sentiment analysis data."""
        if not sentiment_results:
            return []
        
        return [
            {
                "text": result.text,
                "sentiment": result.sentiment.value if hasattr(result.sentiment, 'value') else str(result.sentiment),
                "confidence": result.confidence,
                "start_time": result.start,
                "end_time": result.end,
                "speaker": getattr(result, 'speaker', None)
            }
            for result in sentiment_results
        ]
    
    def _extract_entity_data(self, entities) -> List[Dict]:
        """Extract entity detection data."""
        if not entities:
            return []
        
        return [
            {
                "text": entity.text,
                "entity_type": entity.entity_type.value if hasattr(entity.entity_type, 'value') else str(entity.entity_type),
                "start_time": entity.start,
                "end_time": entity.end
            }
            for entity in entities
        ]
    
    def _extract_topic_data(self, iab_categories) -> Dict:
        """Extract topic detection data."""
        if not iab_categories:
            return {}
        
        data = {
            "summary": dict(iab_categories.summary) if hasattr(iab_categories, 'summary') else {},
            "results": []
        }
        
        if hasattr(iab_categories, 'results'):
            data["results"] = [
                {
                    "text": result.text,
                    "labels": [
                        {"label": label.label, "relevance": label.relevance}
                        for label in result.labels
                    ],
                    "start_time": result.timestamp.start,
                    "end_time": result.timestamp.end
                }
                for result in iab_categories.results
            ]
        
        return data
    
    def _extract_content_safety_data(self, content_safety) -> Dict:
        """Extract content safety data."""
        if not content_safety:
            return {}
        
        data = {
            "summary": dict(content_safety.summary) if hasattr(content_safety, 'summary') else {},
            "results": []
        }
        
        if hasattr(content_safety, 'results'):
            data["results"] = [
                {
                    "text": result.text,
                    "labels": [
                        {
                            "label": label.label,
                            "confidence": label.confidence,
                            "severity": getattr(label, 'severity', None)
                        }
                        for label in result.labels
                    ],
                    "start_time": result.timestamp.start,
                    "end_time": result.timestamp.end
                }
                for result in content_safety.results
            ]
        
        return data
    
    def _extract_highlights_data(self, auto_highlights) -> List[Dict]:
        """Extract auto highlights data."""
        if not auto_highlights or not hasattr(auto_highlights, 'results'):
            return []
        
        return [
            {
                "text": result.text,
                "rank": result.rank,
                "count": result.count,
                "timestamps": [
                    {"start_time": ts.start, "end_time": ts.end}
                    for ts in result.timestamps
                ]
            }
            for result in auto_highlights.results
        ]

    def to_dict(self) -> Dict[str, Any]:
        """Serialize the component to a dictionary."""
        return default_to_dict(
            self,
            config=self.config.__dict__,
            api_key="***",  # Don't serialize the actual API key
            polling_interval=self.polling_interval
        )

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "AssemblyAITranscriber":
        """Deserialize the component from a dictionary."""
        config_data = data["init_parameters"].get("config", {})
        config = AudioProcessingConfig(**config_data)
        
        return default_from_dict(
            cls,
            data,
            config=config
        )


@component
class SmartAudioProcessor:
    """
    Advanced audio processor that mimics mew.py's smart document chunking
    but for audio content with speaker awareness and content boundaries.
    """
    
    def __init__(
        self,
        assemblyai_transcriber: AssemblyAITranscriber,
        max_chunk_length: int = 1000,
        overlap: int = 100,
        respect_speakers: bool = True,
        respect_chapters: bool = True
    ):
        self.transcriber = assemblyai_transcriber
        self.max_chunk_length = max_chunk_length
        self.overlap = overlap
        self.respect_speakers = respect_speakers
        self.respect_chapters = respect_chapters
    
    @component.output_types(documents=List[Document])
    def run(self, sources: List[Union[str, Path, bytes]]) -> Dict[str, List[Document]]:
        """Process audio with smart chunking like mew.py does for documents."""
        
        # First get the transcription with all features
        transcription_result = self.transcriber.run(sources)
        raw_documents = transcription_result["documents"]
        
        smart_chunks = []
        
        for doc in raw_documents:
            if doc.meta.get("content_type") in ["sentence", "paragraph"]:
                # These are already structured chunks, skip
                continue
            
            # Process main transcript document with smart chunking
            if "content_type" not in doc.meta:
                chunks = self._create_smart_audio_chunks(doc)
                smart_chunks.extend(chunks)
        
        return {"documents": smart_chunks}
    
    def _create_smart_audio_chunks(self, document: Document) -> List[Document]:
        """Create smart chunks from audio transcript similar to mew.py's approach."""
        
        chunks = []
        content = document.content
        metadata = document.meta.copy()
        
        # Check if we have speaker or chapter information
        speaker_data = metadata.get("sentiment_analysis", [])
        chapter_info = content.find("## Chapters") != -1
        
        if self.respect_speakers and speaker_data:
            # Speaker-aware chunking
            chunks = self._chunk_by_speakers(content, metadata, speaker_data)
        elif self.respect_chapters and chapter_info:
            # Chapter-aware chunking
            chunks = self._chunk_by_chapters(content, metadata)
        else:
            # Semantic boundary-aware chunking
            chunks = self._chunk_by_semantic_boundaries(content, metadata)
        
        return chunks
    
    def _chunk_by_speakers(self, content: str, metadata: Dict, speaker_data: List) -> List[Document]:
        """Chunk content based on speaker changes."""
        chunks = []
        lines = content.split('\n')
        
        current_chunk = []
        current_speaker = None
        chunk_id = 0
        
        for line in lines:
            if line.startswith("**Speaker "):
                # New speaker detected
                if current_chunk and current_speaker:
                    # Save previous chunk
                    chunk_content = '\n'.join(current_chunk)
                    if len(chunk_content.strip()) > 0:
                        chunk_metadata = metadata.copy()
                        chunk_metadata.update({
                            "chunk_id": chunk_id,
                            "chunk_type": "speaker_segment",
                            "speaker": current_speaker,
                            "chunk_length": len(chunk_content),
                            "processing_strategy": "speaker_aware"
                        })
                        
                        chunks.append(Document(content=chunk_content, meta=chunk_metadata))
                        chunk_id += 1
                
                # Start new chunk
                current_chunk = [line]
                # Extract speaker info
                if "Speaker " in line:
                    try:
                        current_speaker = line.split("Speaker ")[1].split("**")[0]
                    except:
                        current_speaker = "Unknown"
            else:
                current_chunk.append(line)
        
        # Add final chunk
        if current_chunk:
            chunk_content = '\n'.join(current_chunk)
            if len(chunk_content.strip()) > 0:
                chunk_metadata = metadata.copy()
                chunk_metadata.update({
                    "chunk_id": chunk_id,
                    "chunk_type": "speaker_segment",
                    "speaker": current_speaker or "Unknown",
                    "chunk_length": len(chunk_content),
                    "processing_strategy": "speaker_aware"
                })
                
                chunks.append(Document(content=chunk_content, meta=chunk_metadata))
        
        return chunks
    
    def _chunk_by_chapters(self, content: str, metadata: Dict) -> List[Document]:
        """Chunk content based on auto-generated chapters."""
        chunks = []
        
        # Find chapter sections
        sections = content.split("### Chapter")
        
        for i, section in enumerate(sections):
            if i == 0:  # Skip the part before first chapter
                continue
            
            if len(section.strip()) > 0:
                chunk_content = f"### Chapter{section}"
                chunk_metadata = metadata.copy()
                chunk_metadata.update({
                    "chunk_id": i - 1,
                    "chunk_type": "chapter",
                    "chapter_number": i,
                    "chunk_length": len(chunk_content),
                    "processing_strategy": "chapter_aware"
                })
                
                chunks.append(Document(content=chunk_content, meta=chunk_metadata))
        
        return chunks
    
    def _chunk_by_semantic_boundaries(self, content: str, metadata: Dict) -> List[Document]:
        """Chunk content based on semantic boundaries like sentences and paragraphs."""
        chunks = []
        
        # Split by double newlines (paragraph breaks) and other semantic indicators
        sections = content.split('\n\n')
        
        current_chunk = ""
        chunk_id = 0
        
        for section in sections:
            if len(current_chunk) + len(section) > self.max_chunk_length and current_chunk:
                # Save current chunk
                if current_chunk.strip():
                    chunk_metadata = metadata.copy()
                    chunk_metadata.update({
                        "chunk_id": chunk_id,
                        "chunk_type": "semantic_boundary",
                        "chunk_length": len(current_chunk),
                        "processing_strategy": "semantic_aware"
                    })
                    
                    chunks.append(Document(content=current_chunk.strip(), meta=chunk_metadata))
                    chunk_id += 1
                
                current_chunk = section
            else:
                current_chunk += "\n\n" + section if current_chunk else section
        
        # Add final chunk
        if current_chunk.strip():
            chunk_metadata = metadata.copy()
            chunk_metadata.update({
                "chunk_id": chunk_id,
                "chunk_type": "semantic_boundary", 
                "chunk_length": len(current_chunk),
                "processing_strategy": "semantic_aware"
            })
            
            chunks.append(Document(content=current_chunk.strip(), meta=chunk_metadata))
        
        return chunks

    def to_dict(self) -> Dict[str, Any]:
        """Serialize the component to a dictionary."""
        return default_to_dict(
            self,
            assemblyai_transcriber=self.transcriber.to_dict(),
            max_chunk_length=self.max_chunk_length,
            overlap=self.overlap,
            respect_speakers=self.respect_speakers,
            respect_chapters=self.respect_chapters
        )

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "SmartAudioProcessor":
        """Deserialize the component from a dictionary."""
        transcriber_data = data["init_parameters"]["assemblyai_transcriber"]
        transcriber = AssemblyAITranscriber.from_dict(transcriber_data)
        
        return default_from_dict(
            cls,
            data,
            assemblyai_transcriber=transcriber
        )


# Example usage and integration functions
def create_audio_pipeline(
    api_key: Optional[str] = None,
    config: Optional[AudioProcessingConfig] = None
) -> 'Pipeline':
    """
    Create a complete audio processing pipeline with AssemblyAI.
    
    :param api_key: AssemblyAI API key
    :param config: Audio processing configuration
    :return: Configured Haystack pipeline
    """
    from haystack import Pipeline
    from haystack.components.embedders import SentenceTransformersDocumentEmbedder
    from haystack.components.writers import DocumentWriter
    from haystack.document_stores.in_memory import InMemoryDocumentStore
    
    # Create components
    transcriber = AssemblyAITranscriber(api_key=api_key, config=config)
    smart_processor = SmartAudioProcessor(assemblyai_transcriber=transcriber)
    embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
    document_store = InMemoryDocumentStore()
    writer = DocumentWriter(document_store=document_store)
    
    # Create pipeline
    pipeline = Pipeline()
    pipeline.add_component("smart_processor", smart_processor)
    pipeline.add_component("embedder", embedder)
    pipeline.add_component("writer", writer)
    
    # Connect components
    pipeline.connect("smart_processor", "embedder")
    pipeline.connect("embedder", "writer")
    
    return pipeline


def create_advanced_audio_config() -> AudioProcessingConfig:
    """Create an advanced configuration with all AssemblyAI features enabled."""
    return AudioProcessingConfig(
        # Speaker analysis
        speaker_labels=True,
        speakers_expected=None,  # Auto-detect
        
        # Content analysis - all features enabled
        sentiment_analysis=True,
        entity_detection=True,
        iab_categories=True,
        content_safety=True,
        content_safety_confidence=75,
        auto_highlights=True,
        
        # Audio enhancement
        noise_reduction=True,
        automatic_punctuation=True,
        format_text=True,
        
        # Privacy features
        redact_pii=False,  # Can be enabled as needed
        redact_pii_policies=["person_name", "phone_number", "email_address"],
        
        # Custom vocabulary
        custom_spelling={
            "AssemblyAI": ["assembly ai", "assembly AI"],
            "Haystack": ["hay stack"],
            "API": ["api", "A.P.I."]
        },
        custom_vocabulary=["transcription", "speech-to-text", "AI", "machine learning"],
        boost_param="high",
        
        # Output structure
        include_utterances=True,
        include_sentences=True,
        include_paragraphs=True,
        auto_chapters=True,
        summarization=True,
        summary_model="informative",
        summary_type="bullets"
    )


if __name__ == "__main__":
    # Example usage
    if ASSEMBLYAI_AVAILABLE:
        print("AssemblyAI Haystack Integration loaded successfully!")
        print("Available components:")
        print("  - AssemblyAITranscriber")
        print("  - SmartAudioProcessor")
        print("  - AudioProcessingConfig")
        print("  - create_audio_pipeline()")
        print("  - create_advanced_audio_config()")
    else:
        print("AssemblyAI package not available. Install with: pip install assemblyai"

# Summary: What You Were Missing for Audio Processing

## Key Features Now Implemented:

### ✅ **Smart Audio Processing Component**
- **Speaker Diarization**: Identifies and tracks different speakers
- **Timestamp Preservation**: Maintains timing information for each segment
- **Smart Chunking**: Speaker-aware chunking that doesn't split conversations
- **Advanced Metadata**: Confidence scores, duration, speaker counts

### ✅ **Enhanced Citation System**
- **Audio-specific Citations**: Includes speaker information and timestamps
- **Unique Chunk IDs**: Content-based hashing for reliable references
- **Temporal References**: Time-based citations (e.g., "Speaker A at 02:15-03:30")

### ✅ **AssemblyAI Advanced Features**
- **Speaker Labels**: Multi-speaker identification and tracking
- **Auto Punctuation**: Improved readability of transcriptions
- **Language Detection**: Automatic language identification
- **Summary Generation**: AI-powered content summarization
- **Confidence Scoring**: Quality assessment of transcriptions

### ✅ **Haystack Integration**
- **Custom Components**: `SmartAudioProcessor` and `AudioSummaryGenerator`
- **Pipeline Compatibility**: Seamless integration with Haystack pipelines
- **Document Store Integration**: Enhanced documents with rich metadata
- **Batch Processing**: Multiple audio files in single pipeline run

## Comparison: Your MEW.py vs Enhanced Haystack

| Feature | MEW.py | Basic Haystack | Enhanced Haystack |
|---------|--------|---------------|-------------------|
| Speaker Diarization | ✅ | ❌ | ✅ |
| Timestamp Information | ✅ | ❌ | ✅ |
| Smart Chunking | ✅ | ❌ | ✅ |
| Citation System | ✅ | ❌ | ✅ |
| Batch Processing | ✅ | ❌ | ✅ |
| Pipeline Integration | ❌ | ✅ | ✅ |
| Advanced Metadata | ✅ | Basic | ✅ |
| Summary Generation | ✅ | ❌ | ✅ |

## Next Steps:
1. **Local Audio Support**: Add file upload capabilities
2. **Multi-language Processing**: Enhanced language detection
3. **Custom Vocabularies**: Domain-specific transcription improvements
4. **Audio Quality Analysis**: Pre-processing recommendations
5. **Export Capabilities**: Multiple format outputs (JSON, SRT, VTT)